In [2]:
########## Parametros Dinamicas ##########
dataset = 'saxophone'
model = 'transformer'
##########################################
est_kern_path =    f"/home/wilson/Documentos/Experimentos_2026/07_Models_AMT_A2S/metrics/amt/saxophone/kern_clean"
target_kern_path = f"/home/wilson/Documentos/Experimentos_2026/07_Models_AMT_A2S/metrics/amt/target/kern_clean"
method = 'AMT'

In [3]:
from typing import List
import pandas as pd
import os
from tqdm import tqdm

In [4]:
est_midi_path = est_kern_path
midi_path = target_kern_path

In [5]:
def levenshtein(a: List[str], b: List[str]) -> int:
    n, m = len(a), len(b)

    if n > m:
        a, b = b, a
        n, m = m, n

    current = range(n + 1)
    for i in range(1, m + 1):
        previous, current = current, [i] + [0] * n
        for j in range(1, n + 1):
            add, delete = previous[j] + 1, current[j - 1] + 1
            change = previous[j - 1]
            if a[j - 1] != b[i - 1]:
                change = change + 1
            current[j] = min(add, delete, change)

    return current[n]

def compute_ser(y_true: List[List[str]], y_pred: List[List[str]]) -> float:
    ed_acc = 0
    length_acc = 0
    ed_acc += levenshtein(y_true, y_pred)
    length_acc += len(y_true)
    return 100.0 * ed_acc / length_acc

def read_krn(file_path: str) -> List[str]:
    with open(file_path, 'r') as fin:
        lines = fin.read().splitlines()
    if not lines:
        return []
    # Separar cada línea por tabulaciones
    split_lines = [line.split('\t') for line in lines]
    # Transponer filas a columnas
    columns = list(zip(*split_lines))
    # Convertir cada columna en lista
    columns = [list(col) for col in columns]

    out_seq = columns
    if not out_seq:
        return []

    # Tokens especiales
    COL_TOKEN = '<COL>'
    ROW_TOKEN = '<ROW>'

    # Unificar columnas
    unified = []

    # Obtener el número máximo de filas en todas las columnas
    num_rows = max(len(col) for col in out_seq)

    # Recorre cada fila
    for row_idx in range(num_rows):
        # Recorre cada columna
        for col in out_seq:
            if row_idx < len(col):
                unified.append(col[row_idx])
            else:
                unified.append('<EMPTY>')  # Si hay columnas más cortas
            unified.append(COL_TOKEN)
        # Elimina el COL_TOKEN final de la fila y añade ROW_TOKEN
        if unified[-1] == COL_TOKEN:
            unified = unified[:-1]
        unified.append(ROW_TOKEN)

    # Eliminar el último ROW_TOKEN 
    if unified[-1] == ROW_TOKEN:
        unified = unified[:-1]

    # Imprime el resultado
    # print(unified)
    return unified

In [6]:
#path_predict = "/home/grfia/wilson/project_2025_1/A2S/code_and_data/data/quartets/crnn/est_krn/k080-01.058.krn"
#path_target = "/home/grfia/wilson/project_2025_1/A2S/code_and_data/data/quartets/target/krn/k080-01.058.krn"
#krn_predict = read_krn(path_predict)
#krn_target = read_krn(path_target)
#jcompute_ser(krn_target, krn_predict)

In [7]:
# Rutas a las carpetas con archivos MIDI
folders = [est_midi_path, midi_path]

# Proceso para calcular las métricas entre los archivos MIDI
folder_1 = est_midi_path
folder_2 = midi_path

colum_dataset = 'Dataset'
colum_file = 'File'
colum_method = 'Method'
colum_model = 'Model'
colum_ser = 'SER'

columnas = [colum_method, colum_dataset, colum_model, colum_file, colum_ser]

# Crea el DataFrame
df = pd.DataFrame(columns=columnas)

search_files = [f for f in os.listdir(folder_2) if os.path.isfile(os.path.join(folder_2, f))]
missing_files = []
invalid_files = []

# Itera con tqdm para mostrar la barra de progreso
for txt_file in tqdm(search_files, desc="Procesando archivos", unit="archivo"):
    file_int_est = os.path.join(folder_1, txt_file)
    file_int_ref = os.path.join(folder_2, txt_file)
    
    if not os.path.isfile(file_int_est) or not os.path.isfile(file_int_ref):
        missing_files.append(txt_file)
        continue
    
    krn_predict = read_krn(file_int_est)
    krn_target = read_krn(file_int_ref)

    if not krn_predict or not krn_target:
        invalid_files.append(txt_file)
        continue

    value_ser = compute_ser(krn_target, krn_predict)
    
    file_save = txt_file.replace(".krn", ".midi")
    df.loc[len(df)] = [method, dataset, model, file_save, value_ser]

if missing_files:
    print(f"Archivos omitidos por falta de correspondencia: {len(missing_files)}")
    for missing_file in missing_files:
        print(f" - {missing_file}")

if invalid_files:
    print(f"Archivos omitidos por estar vacíos o no ser legibles: {len(invalid_files)}")
    for invalid_file in invalid_files:
        print(f" - {invalid_file}")

# Exportar a archivo Excel
output_dir = os.path.dirname(os.path.normpath(est_midi_path))
ruta_archivo = os.path.join(output_dir, f"SER_Eval_Midi_{model}_{dataset}.xlsx")
#ruta_archivo = 'Results_Magenta.xlsx'  # Nombre del archivo
print(f"Exportando DataFrame a '{ruta_archivo}'...")
df.to_excel(ruta_archivo, index=False)  # index=False elimina la columna de índices
print(f"DataFrame exportado exitosamente como '{ruta_archivo}'.")

Procesando archivos: 100%|██████████| 102/102 [00:00<00:00, 573.49archivo/s]


Exportando DataFrame a '/home/wilson/Documentos/Experimentos_2026/07_Models_AMT_A2S/metrics/amt/saxophone/SER_Eval_Midi_transformer_saxophone.xlsx'...
DataFrame exportado exitosamente como '/home/wilson/Documentos/Experimentos_2026/07_Models_AMT_A2S/metrics/amt/saxophone/SER_Eval_Midi_transformer_saxophone.xlsx'.


In [8]:

# Contar solo archivos, no carpetas
num_est_midi = sum(
    1 for archivo in os.listdir(est_midi_path)
    if os.path.isfile(os.path.join(est_midi_path, archivo))
)

num_midi = sum(
    1 for archivo in os.listdir(midi_path)
    if os.path.isfile(os.path.join(midi_path, archivo))
)

print(f"Archivos en est_midi_path: {num_est_midi}")
print(f"Archivos en midi_path:     {num_midi}")

Archivos en est_midi_path: 102
Archivos en midi_path:     102
